# LIN340 Project Notebook

## Colab Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
%cd /content/drive/MyDrive/LIN340Project

In [ ]:
!pip install transformers peft datasets bert-score rouge-score nltk accelerate sentencepiece stanza

## Preprocessing

In [ ]:
import re, os, random, string, stanza

stanza.download('it', processors='tokenize,pos,lemma,constituency', verbose=False)
nlp = stanza.Pipeline('it', processors='tokenize,pos,lemma,constituency', verbose=False)

PUNCT = string.punctuation + '«»""''–—…'

def remove_punctuation(text):
    return text.translate(str.maketrans('', '', PUNCT))

def read_novel(path):
    with open(path, encoding='latin-1') as f:
        txt = f.read()
    txt = re.sub(r'\n', ' ', txt)
    txt = re.sub(r'\s+', ' ', txt)
    return txt.lower()

TAGS = {'NP': 'NP', 'VP': 'VP', 'PP': 'PP', 'ADJP': 'AP', 'ADVP': 'ADVP'}

def render_tree(node, word_iter):
    if node.is_leaf():
        word = next(word_iter, None)
        if word is None:
            return ''
        return remove_punctuation(word.lemma)
    parts = [p for p in [render_tree(c, word_iter) for c in node.children] if p.strip()]
    text = ' '.join(parts)
    if not text.strip():
        return ''
    if node.label in TAGS:
        tag = TAGS[node.label]
        return f'[{tag}_START] {text} [{tag}_END]'
    return text

def process(text, chunk_size=50000):
    raw_sents, tagged_sents = [], []
    i = 0
    while i < len(text):
        chunk = text[i:i + chunk_size]
        if i + chunk_size < len(text):
            last_space = chunk.rfind(' ')
            if last_space > 0:
                chunk = chunk[:last_space]
        doc = nlp(chunk)
        for sent in doc.sentences:
            lemmas = [l for l in [remove_punctuation(w.lemma) for w in sent.words] if l.strip()]
            raw = ' '.join(lemmas).strip()
            if len(raw) <= 10:
                continue
            raw_sents.append(raw)
            tagged = re.sub(r'\s+', ' ', render_tree(sent.constituency, iter(sent.words))).strip()
            tagged_sents.append(tagged if tagged else raw)
        i += len(chunk)
    return raw_sents, tagged_sents

txt  = read_novel('q.txt')
txt2 = read_novel('guerra_agli_umani.txt')
print(f'q: {len(txt)} chars, guerra: {len(txt2)} chars')

q_raw, q_tagged = process(txt)
print(f'q: {len(q_raw)} sentences')

g_raw, g_tagged = process(txt2)
print(f'guerra: {len(g_raw)} sentences')

combined = list(zip(q_raw + g_raw, q_tagged + g_tagged))
random.seed(42)
random.shuffle(combined)
all_raw    = [x[0] for x in combined]
all_tagged = [x[1] for x in combined]
print(f'total: {len(all_raw)} pairs')

In [ ]:
n = len(all_raw)
i80, i90 = int(n * .8), int(n * .9)

train_raw, val_raw, test_raw = all_raw[:i80], all_raw[i80:i90], all_raw[i90:]
train_tagged, val_tagged, test_tagged = all_tagged[:i80], all_tagged[i80:i90], all_tagged[i90:]
print(f'Train: {len(train_raw)}, Val: {len(val_raw)}, Test: {len(test_raw)}')

os.makedirs('data', exist_ok=True)

for name, sents in [('raw_train', train_raw), ('raw_val', val_raw), ('raw_test', test_raw),
                    ('tagged_train', train_tagged), ('tagged_val', val_tagged), ('tagged_test', test_tagged)]:
    with open(f'data/{name}.txt', 'w', encoding='utf-8') as f:
        f.write('\n'.join(sents))
    print(f'wrote data/{name}.txt')

pairs = []
for sent in test_raw:
    words = sent.split()
    if len(words) < 15:
        continue
    cut = int(len(words) * 2 / 3)
    pairs.append(' '.join(words[:cut]))
    pairs.append(' '.join(words[cut:]))
    if len(pairs) >= 40:
        break

with open('data/eval_prompts.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(pairs))
print(f'wrote data/eval_prompts.txt ({len(pairs)//2} pairs)')

## Training

In [ ]:
!python train.py --mode raw

In [ ]:
!python train.py --mode tagged

## Evaluation

In [ ]:
!python evaluate.py

## Results

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open('results/eval_results.json') as f:
    data = json.load(f)

res = {r["mode"]: r for r in data}
colors = {'raw': '#4C72B0', 'tagged': '#DD8452'}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Raw vs Tagged GPT-2 — Evaluation Metrics')

ax = axes[0]
x = np.arange(4)
ax.bar(x - 0.2, [res['raw'][k] for k in ['bleu_1','bleu_2','bleu_3','bleu_4']], 0.4, label='RAW', color=colors['raw'])
ax.bar(x + 0.2, [res['tagged'][k] for k in ['bleu_1','bleu_2','bleu_3','bleu_4']], 0.4, label='TAGGED', color=colors['tagged'])
ax.set_title('BLEU')
ax.set_xticks(x)
ax.set_xticklabels(['BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4'])
ax.set_ylabel('Score')
ax.legend()

ax = axes[1]
x = np.arange(4)
ax.bar(x - 0.2, [res['raw'][k] for k in ['rouge1','rouge2','rougeL','bertscore_f1']], 0.4, label='RAW', color=colors['raw'])
ax.bar(x + 0.2, [res['tagged'][k] for k in ['rouge1','rouge2','rougeL','bertscore_f1']], 0.4, label='TAGGED', color=colors['tagged'])
ax.set_title('ROUGE & BERTScore F1')
ax.set_xticks(x)
ax.set_xticklabels(['ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'BERTScore F1'])
ax.set_ylabel('Score')
ax.legend()

ax = axes[2]
ax.bar(['RAW', 'TAGGED'], [res['raw']['perplexity'], res['tagged']['perplexity']],
       color=[colors['raw'], colors['tagged']])
ax.set_title('Perplexity (lower = better)')
ax.set_ylabel('Perplexity')

plt.tight_layout()
plt.savefig('results/metrics_comparison.png')
plt.show()

In [ ]:
import json

with open('results/eval_results.json') as f:
    data = json.load(f)

metrics = ['perplexity', 'bleu_1', 'bleu_2', 'bleu_3', 'bleu_4',
           'rouge1', 'rouge2', 'rougeL', 'bertscore_p', 'bertscore_r', 'bertscore_f1']

print(f"{'Metric':<22}" + '  '.join(r['mode'].upper() for r in data))
print('-' * 50)
for m in metrics:
    print(f'{m:<22}' + '  '.join(f'{r[m]:.4f}' for r in data))

In [ ]:
import json, textwrap

with open('results/eval_results.json') as f:
    data = json.load(f)

hyps = {r['mode']: r['hypotheses'] for r in data}

with open('data/eval_prompts.txt', encoding='utf-8') as f:
    lines = [l.strip() for l in f if l.strip()]
prompts    = lines[0::2]
references = lines[1::2]

for i in range(min(5, len(prompts))):
    print(f'\nExample {i+1}')
    print(f'  PROMPT    : {prompts[i]}')
    print(f'  REFERENCE : {references[i]}')
    for mode in ('raw', 'tagged'):
        if mode not in hyps:
            continue
        text = hyps[mode][i]
        wrapped = textwrap.fill(text, width=88, initial_indent=f'  {mode.upper():<10}: ',
                                subsequent_indent=' ' * 14)
        print(wrapped)
    print('-' * 88)